# 🏆 Notebook 4｜迷你專案：kNN＋樸素貝氏＋貝氏網路訊息傳遞

> 對應講義 Part 5＋Demo 3/4。三連發把本章「能動手」的部分收尾。

## 任務 1｜kNN 分類器（含 k 的選擇與自評錯誤率）

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(3)
N = 120
X = np.concatenate([rng.normal([2,2], 1.0, (N,2)), rng.normal([6,5], 1.1, (N,2))])
y = np.array([0]*N + [1]*N)

def knn(Xtr, ytr, pt, k):
    d = np.linalg.norm(Xtr - pt, axis=1)
    kn = np.argsort(d)[:k]
    return int(np.bincount(ytr[kn]).argmax())

errs, ks = [], range(1, 16, 2)
for k in ks:
    e = sum(knn(X, y, p, k) != c for p, c in zip(X, y)) / (2*N)
    errs.append(e)
plt.figure(figsize=(6, 3.5)); plt.plot(list(ks), errs, 'o-')
plt.xlabel('k'); plt.ylabel('自評錯誤率'); plt.title('kNN：k 越大通常越平滑（小樣本留意過擬合）'); plt.grid(alpha=.3); plt.show()

# 決策區（粗略網格）—— Voronoi 的推廣
def grid_cls(k=9, res=140):
    g = np.linspace(-1, 9, res); xx, yy = np.meshgrid(g, g)
    C = np.array([knn(X, y, p, k) for p in np.stack([xx.ravel(), yy.ravel()], 1)]).reshape(res, res)
    return xx, yy, C
xx, yy, C = grid_cls()
plt.figure(figsize=(5,5)); plt.contourf(xx, yy, C, alpha=.3, colors=['#2563eb','#dc2626'])
plt.scatter(X[y==0,0], X[y==0,1], c='#2563eb', s=15); plt.scatter(X[y==1,0], X[y==1,1], c='#dc2626', s=15)
plt.title('kNN 決策區域（k=9）'); plt.show()

## 任務 2｜樸素貝氏（二元特徵，課本 Example 2.10）

> 假設特徵獨立：只需 2l 個參數；且對數似然比是線性函數（2.120）。

In [ ]:
rng = np.random.default_rng(5)
l, N = 8, 2000
p1 = rng.uniform(0.2, 0.8, l); q1 = rng.uniform(0.2, 0.8, l)   # P(xi=1|ω1)=pi、P(xi=1|ω2)=qi
X1 = (rng.random((N, l)) < p1).astype(int); X2 = (rng.random((N, l)) < q1).astype(int)
X = np.vstack([X1, X2]); y = np.array([0]*N + [1]*N)

def nb_predict(x):  # 對數似然比
    LL1 = np.where(x == 1, np.log(p1), np.log(1 - p1)).sum()
    LL2 = np.where(x == 1, np.log(q1), np.log(1 - q1)).sum()
    return 0 if LL1 > LL2 else 1
acc = np.mean([nb_predict(xx) == c for xx, c in zip(X, y)])
print(f'樸素貝氏（l={l} 二元特徵）自評準確率 = {acc*100:.1f}%')

## 任務 3｜貝氏網路訊息傳遞：課本 Fig 2.29（x→y→z→w）

In [ ]:
# 表格指定（課本 Example 2.11）
Px1 = 0.60; Py = {'x1': 0.40, 'x0': 0.30}; Pz = {'y1': 0.25, 'y0': 0.60}; Pw = {'z1': 0.45, 'z0': 0.30}
Px0 = 1 - Px1
Py1 = Py['x1']*Px1 + Py['x0']*Px0; Py0 = 1 - Py1
Pz1 = Pz['y1']*Py1 + Pz['y0']*Py0; Pz0 = 1 - Pz1
Pw1 = Pw['z1']*Pz1 + Pw['z0']*Pz0; Pw0 = 1 - Pw1

# 向下傳：證據 x1
Pz1_x1 = Pz['y1']*Py['x1'] + Pz['y0']*(1 - Py['x1'])
Pw0_x1 = (1-Pw['z1'])*Pz1_x1 + (1-Pw['z0'])*(1 - Pz1_x1)
print(f'P(z1|x1) = {Pz1_x1:.3f}  （課本 0.46）')
print(f'P(w0|x1) = {Pw0_x1:.3f}  （課本 0.63）')

# 向上傳：證據 w1
Pz1_w1 = Pw['z1']*Pz1 / Pw1
print(f'P(z1|w1) = {Pz1_w1:.3f}  （課本 0.57）   P(w1) = {Pw1:.3f}（課本 0.37）')

### 🏆 完成度對照
| 概念 | 你做了 |
|---|---|
| kNN／Voronoi 決策區（2.6） | 任務 1：k 掃描＋決策區 |
| 樸素貝氏（2.5.7） | 任務 2：l=8 二元特徵，對數似然比 |
| 貝氏網路訊息傳遞（2.7） | 任務 3：向下/向上傳播與課本一致的數字 |

> 下一步：進 Ch3（線性分類器）——本章的 LDA/kNN 就是線性/非線性的第一批例子。